In [150]:
# web_research.ipynb
# Generates JS data modules for Research page visualizations
# 
# Inputs (from 3_openalex_pipeline.ipynb):
#   - sec3a_institutions_map.csv      → institutionsMapData.js
#   - institution_edges.csv           → institutionsCollaborationData.js  
#   - sec3b_institution_macrocategory_counts.csv → institutionsTopicsData.js
#
# Outputs (to site/data/research/):
#   - institutionsMapData.js
#   - institutionsCollaborationData.js
#   - institutionsTopicsData.js

In [151]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# =============================================================================
# PATHS
# =============================================================================
TABLES_DIR = Path("../data/processed/outputs/openalex_notebook_outputs/tables")
OUT_DIR = Path("../site/data/research")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Input files (from 3_openalex_pipeline.ipynb)
INSTITUTIONS_MAP_CSV = TABLES_DIR / "sec3a_institutions_map.csv"
INSTITUTION_EDGES_CSV = TABLES_DIR / "institution_edges.csv"
INSTITUTION_MACRO_CSV = TABLES_DIR / "sec3b_institution_macrocategory_counts.csv"

# =============================================================================
# SHARED CONSTANTS
# =============================================================================
REGION_COLORS = {
    "North America": "#3b82f6",
    "Europe": "#10b981",
    "Asia": "#f59e0b",
    "Oceania": "#8b5cf6",
    "South America": "#ef4444",
    "Africa": "#ec4899",
    "Other": "#9ca3af",
}

TOPIC_COLORS = {
    "High-Dimensional Data Analysis": "#8E24AA",
    "Graph Visualization & Text Mining": "#D32F2F",
    "Volume Rendering & Immersive Tech": "#1E88E5",
    "Visual Programming & ML": "#388E3C",
    "Social & Biomedical Analytics": "#00897B",
    "Imaging & Display Technology": "#FB8C00",
    "Causality & Temporal Analysis": "#C2185B",
    "Perception & Uncertainty Vis": "#5E35B1",
    "Topological Data Analysis": "#F57C00",
    "Network Security & Anomaltic": "#455A64",
    "Geospatial & Seismic Vis": "#6D4C41",
    "Molecular Simulation": "#E91E63",
}

# Short name mappings for chord diagram
SHORT_NAME_MAP = {
    # North America
    "University of Utah": "Utah",
    "University of California, Davis": "UC Davis",
    "Stony Brook University": "Stony Brook",
    "Georgia Institute of Technology": "Georgia Tech",
    "Ohio State University": "Ohio State",
    "University of Maryland, College Park": "UMD",
    "Lawrence Livermore National Laboratory": "LLNL",
    "Purdue University West Lafayette": "Purdue",
    "Stanford University": "Stanford",
    "Harvard University": "Harvard",
    "Arizona State University": "ASU",
    "University of Washington": "UW",
    "Simon Fraser University": "SFU",
    "Virginia Polytechnic Institute and State University": "Virginia Tech",
    "Microsoft": "Microsoft",
    "New York University": "NYU",
    "University of North Carolina at Chapel Hill": "UNC",
    "State University of New York": "SUNY",
    "University of North Carolina at Charlotte": "UNC Charlotte",
    "University of British Columbia": "UBC",
    "Ames Research Center": "NASA Ames",
    "University of Calgary": "UCalgary",
    "IBM Research - Thomas J. Watson Research Center": "IBM Watson",
    "Carnegie Mellon University": "CMU",
    "Harvard University Press": "HUP",
    "IBM (United States)": "IBM (US)",
    "Northwestern University": "Northwestern",
    "University of California, Berkeley": "UC Berkeley",
    "Mississippi State University": "MSU",
    "University of Toronto": "UofT",
    "Los Alamos National Laboratory": "LANL",
    "Pacific Northwest National Laboratory": "PNNL",
    # Asia
    "Hong Kong University of Science and Technology": "HKUST",
    "University of Hong Kong": "HKU",
    "Zhejiang University": "ZJU",
    "Tsinghua University": "Tsinghua",
    "Peking University": "PKU",
    # Europe
    "TU Wien": "TU Wien",
    "University of Stuttgart": "Stuttgart",
    "University of Konstanz": "Konstanz",
    "VRVis GmbH": "VRVis",
    "Eindhoven University of Technology": "TU/e",
    "University of Bergen": "UiB",
    "ETH Zurich": "ETH",
    "University of Kaiserslautern": "TU KL",
    "Linköping University": "LiU",
    "Institut national de recherche en informatique et en automatique": "INRIA (FR))",
    "Centre National de la Recherche Scientifique": "CNRS (FR)",
    "Delft University of Technology": "TU Delft",
    "City, University of London": "City, UoL",
    "Université Paris-Sud": "Paris-Sud",
    "Microsoft Research (United Kingdom)": "Microsoft Research (UK)",
    "Microsoft (United States)": "Microsoft (US)",
    "VRVis GmbH (Austria)": "VRVis (AT)",
    # Oceania
    "Monash University": "Monash",
}

# Americas split
NORTH_AMERICA_CODES = {"US","CA","MX","GT","BZ","SV","HN","NI","CR","PA","CU","DO","HT","JM","BS","BB","TT"}
SOUTH_AMERICA_CODES = {"BR","AR","CL","CO","PE","VE","EC","BO","PY","UY","GY","SR","GF"}

def safe_str(x) -> str:
    """Convert to string, handling NaN."""
    return "" if pd.isna(x) else str(x).strip()

def map_region(world_region: str, country_code: str, lat: float = None) -> str:
    """Map world_region to display region, splitting Americas."""
    wr = safe_str(world_region)
    cc = safe_str(country_code).upper()
    
    if wr == "Americas":
        if cc in NORTH_AMERICA_CODES:
            return "North America"
        if cc in SOUTH_AMERICA_CODES:
            return "South America"
        # Fallback by latitude
        if lat is not None and not pd.isna(lat):
            return "South America" if float(lat) < 15 else "North America"
        return "North America"
    
    if wr in {"Europe", "Asia", "Oceania", "Africa"}:
        return wr
    
    return "Other"

def short_name(full_name: str) -> str:
    """Shorten institution name for display."""
    name = safe_str(full_name)
    if not name:
        return name
    
    # Check explicit mapping first
    if name in SHORT_NAME_MAP:
        return SHORT_NAME_MAP[name]
    
    # Generic rules as fallback
    # Harvard University -> Harvard
    if name.endswith(" University") and not name.startswith("University of "):
        return name.replace(" University", "")
    
    # University of California, Davis -> UC Davis
    if name.startswith("University of California, "):
        return "UC " + name.replace("University of California, ", "")
    
    # University of Utah -> Utah
    if name.startswith("University of "):
        return name.replace("University of ", "")
    
    return name

def write_js(path: Path, exports: dict, header: str = ""):
    """Write JS module with exports."""
    lines = []
    if header:
        lines.extend(["/**", f" * {header}", " */", ""])
    
    for name, value in exports.items():
        lines.append(f"export const {name} = {json.dumps(value, ensure_ascii=False, indent=2)};")
        lines.append("")
    
    path.write_text("\n".join(lines), encoding="utf-8")
    print(f"✅ {path.name}")

print("Configuration loaded")
print(f"Input: {TABLES_DIR}")
print(f"Output: {OUT_DIR}")
print(f"Short name mappings: {len(SHORT_NAME_MAP)}")

Configuration loaded
Input: ../data/processed/outputs/openalex_notebook_outputs/tables
Output: ../site/data/research
Short name mappings: 55


## 1. institutionsMapData.js
Bubble map showing global distribution of research institutions.

In [152]:
# Load institutions map data
# Columns: institution_id, institution_name, country_code, country_name, world_region, 
#          latitude, longitude, city, paper_count

df = pd.read_csv(INSTITUTIONS_MAP_CSV)

# Clean numeric columns
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["paper_count"] = pd.to_numeric(df["paper_count"], errors="coerce").fillna(0).astype(int)

# Drop rows without coordinates
df = df.dropna(subset=["latitude", "longitude"]).copy()

# Add region_final
df["region"] = df.apply(
    lambda r: map_region(r["world_region"], r["country_code"], r["latitude"]), 
    axis=1
)

# Build city label
df["city_label"] = df.apply(
    lambda r: f"{safe_str(r['city'])}, {safe_str(r['country_code'])}" 
              if safe_str(r['city']) else safe_str(r['country_name']),
    axis=1
)

# Sort by papers desc
df = df.sort_values("paper_count", ascending=False)

# Build data array
institutions = []
for _, r in df.iterrows():
    institutions.append({
        "name": safe_str(r["institution_name"]),
        "country": safe_str(r["country_name"]),
        "lat": float(r["latitude"]),
        "lon": float(r["longitude"]),
        "papers": int(r["paper_count"]),
        "region": r["region"],
        "city": r["city_label"],
    })

# Stats
total_papers = sum(x["papers"] for x in institutions)
region_counts = {}
for x in institutions:
    region_counts[x["region"]] = region_counts.get(x["region"], 0) + 1

top = institutions[0] if institutions else None

institutionsMapStats = {
    "totalInstitutions": len(institutions),
    "totalPapers": total_papers,
    "regions": region_counts,
    "topInstitution": top["name"] if top else None,
    "topInstitutionPapers": top["papers"] if top else 0,
}

# Write JS
write_js(
    OUT_DIR / "institutionsMapData.js",
    {
        "institutionsMapData": institutions,
        "institutionsMapStats": institutionsMapStats,
        "regionColors": REGION_COLORS,
    },
    header="data/research/institutionsMapData.js - Geographic bubble map data"
)

print(f"Institutions: {len(institutions)}")
print(f"Top: {institutionsMapStats['topInstitution']} ({institutionsMapStats['topInstitutionPapers']} papers)")

✅ institutionsMapData.js
Institutions: 1262
Top: University of Utah (152 papers)


In [153]:
TOP_N = 50  # Number of institutions for chord diagram

# Region order for grouping (clockwise on chord diagram)
REGION_ORDER = {
    "North America": 0,
    "Europe": 1,
    "Asia": 2,
    "Oceania": 3,
    "South America": 4,
    "Africa": 5,
    "Other": 6,
}

# Load data
inst_df = pd.read_csv(INSTITUTIONS_MAP_CSV)
edges_df = pd.read_csv(INSTITUTION_EDGES_CSV)

# Normalize institution IDs
inst_df["institution_id"] = inst_df["institution_id"].astype(str)
inst_df["paper_count"] = pd.to_numeric(inst_df["paper_count"], errors="coerce").fillna(0).astype(int)

# Add region
inst_df["region"] = inst_df.apply(
    lambda r: map_region(r["world_region"], r["country_code"], r.get("latitude")),
    axis=1
)

# Select top N by paper count
top_inst = inst_df.nlargest(TOP_N, "paper_count").copy()
top_ids = set(top_inst["institution_id"].tolist())

# Build lookup
id_to_info = {
    r["institution_id"]: {
        "name": safe_str(r["institution_name"]),
        "short": short_name(r["institution_name"]),
        "region": r["region"],
        "papers": int(r["paper_count"]),
    }
    for _, r in top_inst.iterrows()
}

# Ensure unique short names
short_counts = {}
for info in id_to_info.values():
    short_counts[info["short"]] = short_counts.get(info["short"], 0) + 1

for iid, info in id_to_info.items():
    if short_counts[info["short"]] > 1:
        row = top_inst[top_inst["institution_id"] == iid].iloc[0]
        cc = safe_str(row.get("country_code"))
        if cc:
            info["short"] = f"{info['short']} ({cc})"

# ============================================================
# SORT BY REGION then by papers (descending) within each region
# ============================================================
top_inst["region_order"] = top_inst["region"].map(REGION_ORDER).fillna(99)
top_inst = top_inst.sort_values(
    ["region_order", "paper_count"], 
    ascending=[True, False]
).reset_index(drop=True)

# Build ordered lists (now sorted by region)
ordered_ids = top_inst["institution_id"].tolist()
institutions = [id_to_info[iid]["short"] for iid in ordered_ids]
institutionsFull = [id_to_info[iid]["name"] for iid in ordered_ids]
institutionPapers = [id_to_info[iid]["papers"] for iid in ordered_ids]
institutionRegions = [id_to_info[iid]["region"] for iid in ordered_ids]

id_to_idx = {iid: i for i, iid in enumerate(ordered_ids)}

# Filter edges to top institutions
edges_df["institution_a"] = edges_df["institution_a"].astype(str)
edges_df["institution_b"] = edges_df["institution_b"].astype(str)
edges_df["weight"] = pd.to_numeric(edges_df["weight"], errors="coerce").fillna(0).astype(int)

edges_sel = edges_df[
    edges_df["institution_a"].isin(top_ids) &
    edges_df["institution_b"].isin(top_ids) &
    (edges_df["weight"] > 0)
]

# Build symmetric matrix
n = len(institutions)
matrix = np.zeros((n, n), dtype=int)

for _, r in edges_sel.iterrows():
    i = id_to_idx.get(r["institution_a"])
    j = id_to_idx.get(r["institution_b"])
    if i is not None and j is not None and i != j:
        w = int(r["weight"])
        matrix[i, j] += w
        matrix[j, i] += w

np.fill_diagonal(matrix, 0)

# Stats
totalCollaborations = int(np.triu(matrix, 1).sum())
avgCollab = float(matrix.sum(axis=1).mean()) if n else 0.0

# Strongest pair
strongestPair = {"inst1": None, "inst2": None, "papers": 0}
if n > 1 and totalCollaborations > 0:
    tri = np.triu(matrix, 1)
    idx = np.unravel_index(np.argmax(tri), tri.shape)
    strongestPair = {
        "inst1": institutions[idx[0]],
        "inst2": institutions[idx[1]],
        "papers": int(tri[idx])
    }

# Regional clusters
regionalClusters = {}
for name, reg in zip(institutions, institutionRegions):
    regionalClusters.setdefault(reg, []).append(name)

# Cross-regional rate
cross_sum = sum(
    matrix[i, j]
    for i in range(n) for j in range(i + 1, n)
    if matrix[i, j] > 0 and institutionRegions[i] != institutionRegions[j]
)
crossRegionalRate = cross_sum / totalCollaborations if totalCollaborations else 0.0

institutionsCollaborationData = {
    "institutions": institutions,
    "institutionsFull": institutionsFull,
    "institutionPapers": institutionPapers,
    "institutionRegions": institutionRegions,
    "matrix": matrix.tolist(),
}

institutionsCollaborationStats = {
    "totalInstitutions": n,
    "totalCollaborations": totalCollaborations,
    "avgCollaborationsPerInstitution": round(avgCollab, 2),
    "strongestPair": strongestPair,
    "regionalClusters": regionalClusters,
    "crossRegionalRate": round(crossRegionalRate, 4),
}

# Write JS
write_js(
    OUT_DIR / "institutionsCollaborationData.js",
    {
        "institutionsCollaborationData": institutionsCollaborationData,
        "institutionsCollaborationStats": institutionsCollaborationStats,
        "regionColorsChord": REGION_COLORS,
    },
    header="data/research/institutionsCollaborationData.js - Chord diagram data"
)

print(f"Institutions: {n}")
print(f"Total collaborations: {totalCollaborations}")
print(f"Strongest: {strongestPair}")
print(f"\nOrder by region:")
for reg in sorted(regionalClusters.keys(), key=lambda r: REGION_ORDER.get(r, 99)):
    print(f"  {reg}: {len(regionalClusters[reg])} institutions")

✅ institutionsCollaborationData.js
Institutions: 50
Total collaborations: 949
Strongest: {'inst1': 'HKU', 'inst2': 'HKUST', 'papers': 105}

Order by region:
  North America: 29 institutions
  Europe: 15 institutions
  Asia: 5 institutions
  Oceania: 1 institutions


## 3. institutionsTopicsData.js
Sankey diagram connecting institutions to macro-categories.

In [157]:
TOP_N_SANKEY = 20  # Number of institutions for sankey
MIN_LINK_VALUE = 2  # Minimum papers for a link

# Region order for sorting (same as chord diagram)
REGION_ORDER = {
    "North America": 0,
    "Europe": 1,
    "Asia": 2,
    "Oceania": 3,
    "South America": 4,
    "Africa": 5,
    "Other": 6,
}

# Load data
inst_df = pd.read_csv(INSTITUTIONS_MAP_CSV)
macro_df = pd.read_csv(INSTITUTION_MACRO_CSV)

# Normalize
inst_df["institution_id"] = inst_df["institution_id"].astype(str)
inst_df["paper_count"] = pd.to_numeric(inst_df["paper_count"], errors="coerce").fillna(0).astype(int)

# Add region to institutions
inst_df["region"] = inst_df.apply(
    lambda r: map_region(r["world_region"], r["country_code"], r.get("latitude")),
    axis=1
)

macro_df["institution_id"] = macro_df["institution_id"].astype(str)
macro_df["count"] = pd.to_numeric(macro_df["count"], errors="coerce").fillna(0).astype(int)

# Get top institutions by paper count
top_inst = inst_df.nlargest(TOP_N_SANKEY, "paper_count").copy()
top_ids = set(top_inst["institution_id"])

# ============================================================
# SORT BY REGION then by papers (descending) within each region
# ============================================================
top_inst["region_order"] = top_inst["region"].map(REGION_ORDER).fillna(99)
top_inst = top_inst.sort_values(
    ["region_order", "paper_count"], 
    ascending=[True, False]
).reset_index(drop=True)

# Build ordered list of institution IDs (preserves sort order)
ordered_inst_ids = top_inst["institution_id"].tolist()

# Build institution info lookup (with region!)
inst_info = {
    r["institution_id"]: {
        "name": safe_str(r["institution_name"]),
        "short": short_name(r["institution_name"]),
        "papers": int(r["paper_count"]),
        "region": r["region"],
    }
    for _, r in top_inst.iterrows()
}

# Filter macro_df to top institutions
macro_sel = macro_df[macro_df["institution_id"].isin(top_ids)].copy()

# Get unique topics
topics = sorted(macro_sel["macro_category"].dropna().unique())

# Build nodes
nodes = []

# Institution nodes (left) - SORTED BY REGION
for iid in ordered_inst_ids:
    info = inst_info[iid]
    nodes.append({
        "id": info["short"],
        "type": "institution",
        "fullName": info["name"],
        "totalPapers": info["papers"],
        "region": info["region"],
    })

# Topic nodes (right)
for topic in topics:
    total = macro_sel[macro_sel["macro_category"] == topic]["count"].sum()
    nodes.append({
        "id": topic,
        "type": "topic",
        "category": topic,
        "totalPapers": int(total),
    })

# Build links
links = []
strongest = {"institution": None, "topic": None, "papers": 0}

for _, r in macro_sel.iterrows():
    iid = r["institution_id"]
    topic = r["macro_category"]
    count = int(r["count"])
    
    if pd.isna(topic) or count < MIN_LINK_VALUE:
        continue
    
    source_name = inst_info[iid]["short"]
    
    links.append({
        "source": source_name,
        "target": topic,
        "value": count,
    })
    
    if count > strongest["papers"]:
        strongest = {"institution": source_name, "topic": topic, "papers": count}

# Get present regions for legend
present_regions = sorted(
    set(inst_info[iid]["region"] for iid in ordered_inst_ids),
    key=lambda r: REGION_ORDER.get(r, 99)
)

# Stats
institutionsTopicsStats = {
    "topInstitutions": len(top_ids),
    "topTopics": len(topics),
    "totalConnections": len(links),
    "strongestConnection": strongest,
    "presentRegions": present_regions,
}

institutionsTopicsData = {
    "nodes": nodes,
    "links": links,
}

# Write JS
write_js(
    OUT_DIR / "institutionsTopicsData.js",
    {
        "institutionsTopicsData": institutionsTopicsData,
        "institutionsTopicsStats": institutionsTopicsStats,
        "topicColors": TOPIC_COLORS,
        "regionColors": REGION_COLORS,
    },
    header="data/research/institutionsTopicsData.js - Sankey diagram data"
)

print(f"Institutions: {len(top_ids)}, Topics: {len(topics)}, Links: {len(links)}")
print(f"Strongest: {strongest}")
print(f"\n📊 Institutions ordered by region:")
for reg in present_regions:
    names = [inst_info[iid]["short"] for iid in ordered_inst_ids if inst_info[iid]["region"] == reg]
    print(f"  {reg}: {', '.join(names)}")

✅ institutionsTopicsData.js
Institutions: 20, Topics: 12, Links: 164
Strongest: {'institution': 'Utah', 'topic': 'Volume Rendering & Immersive Tech', 'papers': 57}

📊 Institutions ordered by region:
  North America: Utah, UC Davis, Stony Brook, Georgia Tech, The Ohio State, UMD, Purdue, LLNL, Harvard, Stanford
  Europe: TU Wien, Stuttgart, Konstanz, INRIA (FR)), VRVis (AT), TU/e, CNRS (FR)
  Asia: HKU, HKUST, ZJU
